In [17]:
import pandas as pd
from pathlib import Path

results_dir = Path("results")

files = [
    results_dir / "baseline_results.csv",
    results_dir / "densenet_results.csv",
    results_dir / "mobilenet_results.csv",
    results_dir / "efficientnet_results.csv",
    results_dir / "mobilenetv3_results.csv",
    results_dir / "squeezenet_results.csv",
    results_dir / "squeezenet_screen_results.csv",
]

dfs = [pd.read_csv(f) for f in files]
results_df = pd.concat(dfs, ignore_index=True)

results_df


,Model,Flip,Class Weights,Threshold,AUROC,AUPRC,Accuracy,Macro F1,Weighted F1,Sensitivity,Specificity,Precision,TN,FP,FN,TP,Mode,Params
0,Baseline CNN,0.0,0.0,0.575,0.928359,0.951591,0.825321,0.795875,0.815257,0.964103,0.594017,0.798301,139,95,14,376,Screening,4828481
1,DenseNet121 Stage2,NaN,NaN,0.380,0.965374,0.977245,0.838141,0.808642,0.827425,0.984615,0.594017,0.801670,139,95,6,384,Screening,7038529
2,MobileNetV2,NaN,NaN,0.330,0.966563,0.976543,0.833333,0.800854,0.820960,0.989744,0.572650,0.794239,134,100,4,386,Screening,2259265
3,MobileNetV2,NaN,NaN,0.860,0.966563,0.976543,0.884615,0.869559,0.880638,0.979487,0.726496,0.856502,170,64,8,382,Diagnostic,2259265
4,EfficientNetB0,NaN,NaN,0.250,0.938900,0.959546,0.804487,0.761070,0.786533,0.984615,0.504274,0.768000,118,116,6,384,Screening,4050852
5,EfficientNetB0,NaN,NaN,0.690,0.938900,0.959546,0.871795,0.860714,0.870536,0.923077,0.786325,0.878049,184,50,30,360,Diagnostic,4050852
6,MobileNetV3Large,NaN,NaN,0.320,0.968508,0.980941,0.842949,0.814049,0.832375,0.989744,0.598291,0.804167,140,94,4,386,Screening,2997313
7,squeezenet,NaN,NaN,0.200,0.922233,0.954729,0.724359,0.629290,0.676223,0.984615,0.290598,0.698182,68,166,6,384,NaN,735937
8,squeezenet_threshold_0.35,NaN,NaN,0.350,0.941672,0.963456,0.794872,0.744981,0.773180,0.989744,0.470085,0.756863,110,124,4,386,Screening,735937


In [18]:
quality_table = (
    results_df.sort_values(["Model", "Mode"])
    .drop_duplicates(subset=["Model"])[["Model", "Params", "AUROC", "AUPRC"]]
    .assign(Params_M=lambda d: (d["Params"] / 1e6).round(2))
    .drop(columns=["Params"])
    .sort_values("AUPRC", ascending=False)
)

quality_table


,Model,AUROC,AUPRC,Params_M
6,MobileNetV3Large,0.968508,0.980941,3.00
1,DenseNet121 Stage2,0.965374,0.977245,7.04
3,MobileNetV2,0.966563,0.976543,2.26
8,squeezenet_threshold_0.35,0.941672,0.963456,0.74
5,EfficientNetB0,0.938900,0.959546,4.05
7,squeezenet,0.922233,0.954729,0.74
0,Baseline CNN,0.928359,0.951591,4.83


In [19]:
screening_table = (
    results_df[results_df["Mode"] == "Screening"]
    [["Model", "Params", "Threshold", "Sensitivity", "Specificity", "FN", "FP",
      "Precision", "Accuracy", "AUROC", "AUPRC"]]
      .assign(Params_M=lambda d: (d["Params"] / 1e6).round(2))
      .drop(columns=["Params"])
    .sort_values(["Sensitivity", "Specificity"], ascending=[False, False])
)

screening_table


,Model,Threshold,Sensitivity,Specificity,FN,FP,Precision,Accuracy,AUROC,AUPRC,Params_M
6,MobileNetV3Large,0.320,0.989744,0.598291,4,94,0.804167,0.842949,0.968508,0.980941,3.00
2,MobileNetV2,0.330,0.989744,0.572650,4,100,0.794239,0.833333,0.966563,0.976543,2.26
8,squeezenet_threshold_0.35,0.350,0.989744,0.470085,4,124,0.756863,0.794872,0.941672,0.963456,0.74
1,DenseNet121 Stage2,0.380,0.984615,0.594017,6,95,0.801670,0.838141,0.965374,0.977245,7.04
4,EfficientNetB0,0.250,0.984615,0.504274,6,116,0.768000,0.804487,0.938900,0.959546,4.05
0,Baseline CNN,0.575,0.964103,0.594017,14,95,0.798301,0.825321,0.928359,0.951591,4.83


In [20]:
screening_table_hr = (
    results_df[results_df["Mode"] == "Screening"]
    [["Model", "Params", "Threshold", "Sensitivity", "Specificity", "FN", "FP",
      "Precision", "Accuracy", "AUROC", "AUPRC"]]
    .assign(
        Params_M=lambda d: (d["Params"] / 1e6).round(2),
        Sensitivity=lambda d: (d["Sensitivity"] * 100).round(2),
        Specificity=lambda d: (d["Specificity"] * 100).round(2),
        Precision=lambda d: (d["Precision"] * 100).round(2),
        Accuracy=lambda d: (d["Accuracy"] * 100).round(2),
        AUROC=lambda d: d["AUROC"].round(2),
        AUPRC=lambda d: d["AUPRC"].round(2),
        Threshold=lambda d: d["Threshold"].round(2),
    )
    .drop(columns=["Params"])
    .rename(columns={
        "Params_M": "Params (M)",
        "Sensitivity": "Sensitivity (%)",
        "Specificity": "Specificity (%)",
        "Precision": "Precision (%)",
        "Accuracy": "Accuracy (%)",
    })
    .sort_values(["Sensitivity (%)", "Specificity (%)"], ascending=[False, False])
)
screening_table_hr

,Model,Threshold,Sensitivity (%),Specificity (%),FN,FP,Precision (%),Accuracy (%),AUROC,AUPRC,Params (M)
6,MobileNetV3Large,0.32,98.97,59.83,4,94,80.42,84.29,0.97,0.98,3.00
2,MobileNetV2,0.33,98.97,57.26,4,100,79.42,83.33,0.97,0.98,2.26
8,squeezenet_threshold_0.35,0.35,98.97,47.01,4,124,75.69,79.49,0.94,0.96,0.74
1,DenseNet121 Stage2,0.38,98.46,59.40,6,95,80.17,83.81,0.97,0.98,7.04
4,EfficientNetB0,0.25,98.46,50.43,6,116,76.80,80.45,0.94,0.96,4.05
0,Baseline CNN,0.57,96.41,59.40,14,95,79.83,82.53,0.93,0.95,4.83


Using screening-oriented thresholds, both transfer learning (DenseNet121) and the lightweight MobileNetV2 substantially outperform the baseline CNN in AUROC/AUPRC. At comparable false-positive levels, DenseNet reduces false negatives from 14 to 6 while keeping FP at 95. MobileNetV2 achieves the highest sensitivity (FN = 4) with a small increase in false positives (FP = 100), making it an excellent lightweight screening candidate.
